# 11 - Comparación: `class_weight` vs. submuestreo

Compara en la misma validación la CNN ponderada del notebook 10 contra una CNN entrenada con submuestreo. Ambos modelos usan el mismo criterio de selección de umbral. El checkpoint ponderado se reutiliza; solo se entrena aquí el control submuestreado.

**No se evalúa el test anterior.** La validación participó de la parada temprana y selección de umbral, por lo que la comparación y sus intervalos pareados son exploratorios, no confirmatorios. Los checkpoints de ambos experimentos se guardan por separado y ninguno reemplaza el modelo de inferencia.

## 1. Preparar las mismas particiones

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, recall_score
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.dataset import LesionDataset, get_transforms
from src.entrenamiento import probabilidades
from src.experimento_class_weight import (
    IMAGES,
    UMBRALES,
    UMBRAL_RECALL,
    entrenar_modelo,
    metricas,
    preparar_particiones,
    seleccionar_umbral,
    subconjunto_balanceado,
)
from src.modelo import crear_modelo
from src.utils import SEED

RUTA_METADATA = ROOT / "data" / "raw" / "HAM10000_metadata.csv"
RUTA_TEST = ROOT / "data" / "processed" / "test.csv"
RUTA_CHECKPOINT_PONDERADO = ROOT / "models" / "cnn_resnet18_class_weight.pt"
RUTA_CHECKPOINT_BALANCEADO = ROOT / "models" / "cnn_resnet18_subsampled_control.pt"
RUTA_REPORTE = ROOT / "reports" / "class_weight_vs_submuestreo.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not RUTA_CHECKPOINT_PONDERADO.exists():
    raise FileNotFoundError(f"No se encontró el checkpoint del notebook 10: {RUTA_CHECKPOINT_PONDERADO}")
print(f"Dispositivo: {device}")

Dispositivo: cpu


In [2]:
train_df, val_df = preparar_particiones(RUTA_METADATA, RUTA_TEST)
train_balanceado = subconjunto_balanceado(train_df)
test_lesions = set(pd.read_csv(RUTA_TEST)["lesion_id"])

assert set(train_df["lesion_id"]).isdisjoint(test_lesions)
assert set(val_df["lesion_id"]).isdisjoint(test_lesions)
assert set(train_df["lesion_id"]).isdisjoint(val_df["lesion_id"])
assert set(train_balanceado["lesion_id"]).issubset(set(train_df["lesion_id"]))

print(f"Entrenamiento class_weight: {len(train_df)} ({train_df['etiqueta'].value_counts().to_dict()})")
print(f"Entrenamiento submuestreado: {len(train_balanceado)} ({train_balanceado['etiqueta'].value_counts().to_dict()})")
print(f"Validación compartida: {len(val_df)}; prevalencia melanoma {val_df['etiqueta'].mean():.1%}")
print(f"Lesiones del test anterior excluidas: {len(test_lesions)}")

Entrenamiento class_weight: 4616 ({0: 4223, 1: 393})
Entrenamiento submuestreado: 786 ({0: 393, 1: 393})
Validación compartida: 1155; prevalencia melanoma 8.5%
Lesiones del test anterior excluidas: 246


## 2. Obtener predicciones de `class_weight`

Se carga el checkpoint ya entrenado en el notebook 10 y se predice sobre la validación compartida.

In [3]:
loader_val = DataLoader(
    LesionDataset(val_df, get_transforms(entrenamiento=False), image_dir=IMAGES),
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

modelo_ponderado = crear_modelo(preentrenado=False).to(device)
modelo_ponderado.load_state_dict(
    torch.load(RUTA_CHECKPOINT_PONDERADO, map_location=device, weights_only=True)
)
prob_ponderado, y_true = probabilidades(modelo_ponderado, loader_val, device)
print(f"Predicciones de class_weight: {len(prob_ponderado)}")

Predicciones de class_weight: 1155


## 3. Entrenar el control con submuestreo

Se conserva la misma partición de validación y la misma receta de entrenamiento solo cambia el conjunto de entrenamiento.

In [4]:
salida_balanceada = entrenar_modelo(
    train_balanceado,
    val_df,
    ponderar_clases=False,
    device=device,
    checkpoint_path=RUTA_CHECKPOINT_BALANCEADO,
)
prob_balanceado = salida_balanceada["probabilidades"]
assert np.array_equal(y_true, salida_balanceada["y_true"])
print(f"Control submuestreado guardado: {RUTA_CHECKPOINT_BALANCEADO.exists()}")

[balanceado F1] época 1/8: train=0.6186, val=0.5651
[balanceado F1] época 2/8: train=0.4994, val=0.6850
[balanceado F1] época 3/8: train=0.4266, val=0.6551
[balanceado F1] época 4/8: train=0.4000, val=0.6796
[balanceado F1] época 5/8: train=0.3917, val=0.6274
[balanceado F1] época 6/8: train=0.3957, val=0.6737
[balanceado F1] época 7/8: train=0.3869, val=0.5883
[balanceado F1] época 8/8: train=0.3590, val=0.5581
[balanceado F2] época 1/12: train=0.3563, val=0.4919
[balanceado F2] época 2/12: train=0.2864, val=0.5071
[balanceado F2] época 3/12: train=0.2679, val=0.5373
[balanceado F2] época 4/12: train=0.2395, val=0.4829
[balanceado F2] época 5/12: train=0.2164, val=0.3872
[balanceado F2] época 6/12: train=0.1728, val=0.6306
[balanceado F2] época 7/12: train=0.1653, val=0.6163
[balanceado F2] época 8/12: train=0.1492, val=0.4898
[balanceado F2] época 9/12: train=0.1650, val=0.6038
Parada temprana.
Control submuestreado guardado: True


## 4. Comparar con el mismo criterio y bootstrap pareado

Para cada modelo se selecciona en la validación el umbral que maximiza F1 macro entre los que alcanzan recall ≥ 0,85. Las diferencias del bootstrap se calculan sobre las mismas imágenes y umbrales ya seleccionados.

In [5]:
umbral_ponderado = seleccionar_umbral(y_true, prob_ponderado)
umbral_balanceado = seleccionar_umbral(y_true, prob_balanceado)
if umbral_ponderado is None or umbral_balanceado is None:
    raise ValueError("Algún modelo no alcanzó el recall mínimo en la grilla de umbrales.")

metricas_ponderado = metricas(y_true, prob_ponderado, umbral_ponderado)
metricas_balanceado = metricas(y_true, prob_balanceado, umbral_balanceado)
pred_ponderado = (prob_ponderado >= umbral_ponderado).astype(int)
pred_balanceado = (prob_balanceado >= umbral_balanceado).astype(int)

rng = np.random.default_rng(SEED)
diferencias = {"recall_melanoma": [], "f1_macro": []}
for _ in range(2000):
    indices = rng.integers(0, len(y_true), size=len(y_true))
    y_boot = y_true[indices]
    pred_w_boot = pred_ponderado[indices]
    pred_b_boot = pred_balanceado[indices]
    diferencias["recall_melanoma"].append(
        recall_score(y_boot, pred_w_boot, zero_division=0)
        - recall_score(y_boot, pred_b_boot, zero_division=0)
    )
    diferencias["f1_macro"].append(
        f1_score(y_boot, pred_w_boot, average="macro", zero_division=0)
        - f1_score(y_boot, pred_b_boot, average="macro", zero_division=0)
    )

diferencias_observadas = {
    "recall_melanoma": metricas_ponderado["recall_melanoma"] - metricas_balanceado["recall_melanoma"],
    "f1_macro": metricas_ponderado["f1_macro"] - metricas_balanceado["f1_macro"],
}
bootstrap = {}
for nombre, valores in diferencias.items():
    bootstrap[nombre] = {
        "diferencia_observada_class_weight_menos_submuestreo": float(diferencias_observadas[nombre]),
        "ic95_pareado": [float(x) for x in np.quantile(valores, [0.025, 0.975])],
    }

reporte = {
    "experimento": "class_weight vs. submuestreo",
    "evaluacion": "validacion exploratoria; test anterior excluido",
    "semilla": SEED,
    "lesiones_test_anterior_excluidas": len(test_lesions),
    "entrenamiento": {
        "class_weight": {
            "imagenes": len(train_df),
            "melanomas": int(train_df["etiqueta"].sum()),
            "nevos": int((train_df["etiqueta"] == 0).sum()),
        },
        "submuestreo": {
            "imagenes": len(train_balanceado),
            "melanomas": int(train_balanceado["etiqueta"].sum()),
            "nevos": int((train_balanceado["etiqueta"] == 0).sum()),
        },
    },
    "validacion": {
        "imagenes": len(val_df),
        "prevalencia_melanoma": float(val_df["etiqueta"].mean()),
    },
    "regla_umbral": {
        "grilla": list(UMBRALES),
        "recall_minimo": UMBRAL_RECALL,
        "criterio": "mayor F1 macro entre umbrales elegibles",
    },
    "resultados": {
        "class_weight": metricas_ponderado,
        "submuestreo": metricas_balanceado,
    },
    "bootstrap_pareado_2000": bootstrap,
    "checkpoints": {
        "class_weight": str(RUTA_CHECKPOINT_PONDERADO.relative_to(ROOT)),
        "submuestreo": str(RUTA_CHECKPOINT_BALANCEADO.relative_to(ROOT)),
    },
}

RUTA_REPORTE.parent.mkdir(parents=True, exist_ok=True)
RUTA_REPORTE.write_text(
    json.dumps(reporte, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
tabla = pd.DataFrame(reporte["resultados"]).T
display(tabla[["umbral", "recall_melanoma", "f1_macro", "especificidad", "ppv_estimado_prevalencia_10pct", "falsos_negativos", "falsos_positivos"]])
print(json.dumps(bootstrap, indent=2, ensure_ascii=False))
print(f"Reporte guardado en {RUTA_REPORTE}")

,umbral,recall_melanoma,f1_macro,especificidad,ppv_estimado_prevalencia_10pct,falsos_negativos,falsos_positivos
class_weight,0.5,0.877551,0.719647,0.860927,0.412149,12,147
submuestreo,0.6,0.857143,0.695508,0.842952,0.3775,14,166


{
  "recall_melanoma": {
    "diferencia_observada_class_weight_menos_submuestreo": 0.020408163265306145,
    "ic95_pareado": [
      -0.043010752688172005,
      0.08333333333333337
    ]
  },
  "f1_macro": {
    "diferencia_observada_class_weight_menos_submuestreo": 0.024139176464640255,
    "ic95_pareado": [
      0.00031779813023489486,
      0.04811798183858289
    ]
  }
}
Reporte guardado en c:\Trabajo final\reports\class_weight_vs_submuestreo.json


La comparación usa una validación compartida ya utilizada para parada temprana y selección de umbral, por lo que sus resultados son exploratorios. No se usó el test anterior.